In [41]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from Database.DB_reader import Database
import datetime as dt
from Utilities.fund_analysis.fund_analysis.utils import spreads_creation
from collections import defaultdict
import scipy.stats as stats

In [42]:
db = Database()

In [43]:
# Fetch all spot data

markets = ['de', 'fr', 'at', 'hu']  # List of market codes; adjust as needed
end_date = dt.datetime.today().replace(hour=0, minute=0, second=0, microsecond=0) + pd.DateOffset(days=2, hours=-1)
start_date = end_date - pd.DateOffset(months=24, hours=-1)

spot_list = []

for market in markets:    
    # If the actual table names are different, adjust the table name accordingly.
    query = f"""
    SELECT *
    FROM "spot"."{market}"
    WHERE datetime >= '{start_date}'
      AND datetime <= '{end_date}'
    """
    spot_list.append(pd.read_sql(query, db.connection_string)[['datetime', 'price']].set_index('datetime').sort_index().rename(columns={'price': market}))

spot = pd.concat(spot_list, axis=1)

In [44]:
# Fetch weekly futures data
markets = ['de', 'fr']

for market in markets:
    # If the actual table names are different, adjust the table name accordingly.
    query = f"""
    SELECT *
    FROM "futures"."{market}"
    WHERE datetime >= '{start_date}'
      AND datetime <= '{end_date}'
      AND 
    """

In [45]:

# Create base and peak data
peak_mask = spreads_creation.base_peak_mask(spot.index, 'peak')
spot_base = spot.resample('W').mean()
spot_base.columns = [f"{a}_base" for a in spot_base.columns]
spot_peak = spot[peak_mask].resample('W').mean()
spot_peak.columns = [f"{a}_peak" for a in spot_peak.columns]
weekly_spot = pd.concat([spot_base, spot_peak],axis=1)
# Compute correlation between base and peak
corr_matrix = weekly_spot.corr()

# Replace the diagonal with the lag-1 autocorrelation computed over the last 20 periods.
for col in weekly_spot.columns:
    # Ensure we have at least 20 observations; otherwise, the autocorrelation may be undefined.
    if len(weekly_spot[col]) >= 52:
        # Use .tail(20) to take the last 20 periods and compute the lag-1 autocorrelation.
        ac = weekly_spot[col].tail(52).autocorr(lag=1)
        corr_matrix.loc[col, col] = ac
    else:
        # If not enough data, you can assign NaN or some other value.
        corr_matrix.loc[col, col] = np.nan

print(corr_matrix)

          de_base   fr_base   at_base   hu_base   de_peak   fr_peak   at_peak  \
de_base  0.349787  0.793925  0.925776  0.750422  0.909428  0.751606  0.894888   
fr_base  0.793925  0.730343  0.844137  0.664378  0.791455  0.966894  0.812351   
at_base  0.925776  0.844137  0.753171  0.822377  0.884713  0.826153  0.946347   
hu_base  0.750422  0.664378  0.822377  0.646280  0.698064  0.642383  0.769795   
de_peak  0.909428  0.791455  0.884713  0.698064  0.456989  0.827517  0.956980   
fr_peak  0.751606  0.966894  0.826153  0.642383  0.827517  0.776513  0.848894   
at_peak  0.894888  0.812351  0.946347  0.769795  0.956980  0.848894  0.621197   
hu_peak  0.682226  0.629344  0.759806  0.957897  0.725254  0.651313  0.784921   

          hu_peak  
de_base  0.682226  
fr_base  0.629344  
at_base  0.759806  
hu_base  0.957897  
de_peak  0.725254  
fr_peak  0.651313  
at_peak  0.784921  
hu_peak  0.584076  


In [46]:
markets = ['de', 'fr', 'at', 'hu']
delivery = ['base', 'peak']
periods = ['Wknd_0', 'Wknd_1', 'W_1', 'W_2', 'W_3']
fcst_date = '2025-05-26'

# Build condition strings for each filter list
market_condition = ", ".join(f"'{m}'" for m in markets)
delivery_condition = ", ".join(f"'{d}'" for d in delivery)
periods_condition = ", ".join(f"'{p}'" for p in periods)

query_spot = f"""
    SELECT *
    FROM "MODEL_forecast_prices"."xgboost_mean"
    WHERE fcst_date = '{fcst_date}'
      AND rel_product IN ({periods_condition})
      AND del_type IN ({delivery_condition})
      AND market IN ({market_condition})
      AND scenario_type IN ('base_col', 'perc_col')
"""

fcst_data = pd.read_sql(query_spot, con=db.connection_string)

# Create a nested defaultdict structure
fcst_dict = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))

for market in markets:
    for period in periods:
        for del_type in delivery:
            # Use parentheses for each condition
            temp = fcst_data.loc[
                (fcst_data['rel_product'] == period) &
                (fcst_data['market'] == market) &
                (fcst_data['del_type'] == del_type)
            ].copy()
            
            # Check that temp is not empty to avoid errors
            if not temp.empty:
                # If you expect only one row, select the first row and convert to list
                fcst_dict[market][period][del_type]['perc'] = temp['value'].loc[temp['value_type'].isin(['10th_perc', '25th_perc',
                                                                                                         '75th_perc', '90th_perc'])].to_list()
                # For the 'base' value, extract the first row's value
                fcst_dict[market][period][del_type]['mean'] = temp['value'].loc[temp['value_type'].isin(['base'])].to_list()
            else:
                # Handle the case where no matching data exists (optional)
                fcst_dict[market][period][del_type]['perc'] = None
                fcst_dict[market][period][del_type]['mean'] = None


In [47]:
# Define the desired index for the DataFrame: the percentiles as numbers and the mean as a string label.
df_index = [10, 25, 75, 90, 'mean']

# This will be our final result: a dictionary with key as period and value as the corresponding DataFrame.
result = {}

# Mapping from original value_type names to the numeric percentile for index mapping.
perc_mapping = {
    '10th_perc': 10,
    '25th_perc': 25,
    '75th_perc': 75,
    '90th_perc': 90
}

# Loop over each period
for period in periods:
    # Temporary dictionary to hold data for each DataFrame column (each combination of market and delivery type)
    col_data = {}
    
    for market in markets:
        for del_type in delivery:
            # Construct a column name combining market and delivery type.
            col_name = f"{market}_{del_type}"
            
            # Filter the fcst_data DataFrame for the current market, period, and delivery type.
            temp = fcst_data.loc[
                (fcst_data['rel_product'] == period) &
                (fcst_data['market'] == market) &
                (fcst_data['del_type'] == del_type)
            ].copy()
            
            # Initialize the data for each row in our eventual DataFrame.
            # The keys 10, 25, 75, 90 will store the percentile values while 'mean' will store the base value.
            data = {10: None, 25: None, 75: None, 90: None, 'mean': None}
            
            if not temp.empty:
                # Loop over each expected percentile type and update data if it exists in the filtered rows.
                for value_type, perc_val in perc_mapping.items():
                    # Filter for current percentile value type.
                    row = temp[temp['value_type'] == value_type]
                    if not row.empty:
                        # If there are multiple rows, you could decide to take the first (or apply an aggregation).
                        data[perc_val] = row['value'].iloc[0]
                
                # For the mean, we check rows where value_type is 'base'
                mean_row = temp[temp['value_type'] == 'base']
                if not mean_row.empty:
                    data['mean'] = mean_row['value'].iloc[0]
            
            # Add the dictionary for this (market, del_type) as a column in our temporary dictionary.
            col_data[col_name] = data
    
    # Construct a DataFrame for the current period.
    # Here, columns come from the keys of col_data, and rows are defined by df_index.
    df = pd.DataFrame(col_data, index=df_index)
    
    # Save the DataFrame in our result dictionary using the period as key.
    result[period] = df


In [48]:
result

{'Wknd_0':        de_base  de_peak   fr_base  fr_peak   at_base  at_peak   hu_base  \
 10    0.132506      NaN -0.144537      NaN  0.157123      NaN  0.182425   
 25    0.482257      NaN -0.013363      NaN  0.475455      NaN  0.458056   
 75    1.245157      NaN  0.303985      NaN  1.105837      NaN  1.114943   
 90    1.457288      NaN  0.394538      NaN  1.263468      NaN  1.288178   
 mean  0.841650      NaN  0.137172      NaN  0.826987      NaN  0.782185   
 
       hu_peak  
 10        NaN  
 25        NaN  
 75        NaN  
 90        NaN  
 mean      NaN  ,
 'Wknd_1':        de_base  de_peak   fr_base  fr_peak   at_base  at_peak   hu_base  \
 10   -0.434932      NaN -0.405708      NaN -0.290887      NaN -0.185365   
 25    0.021198      NaN -0.190661      NaN  0.131492      NaN  0.186639   
 75    1.014318      NaN  0.299947      NaN  0.985666      NaN  1.053838   
 90    1.280700      NaN  0.442453      NaN  1.229267      NaN  1.312275   
 mean  0.479780      NaN  0.039852     

In [49]:
import itertools

full_corr = pd.DataFrame(corr_matrix)

# ----------------------------
# Simulation Parameters
# ----------------------------
n_samples = 1000        # Number of simulation samples
spread_factor = 55      # Scaling factor for the spread
score = -5              # A specific spread value to compute its percentile

# Mapping from forecast percentile names to numeric values for simulation.
perc_mapping = {
    '10th_perc': 10,
    '25th_perc': 25,
    '75th_perc': 75,
    '90th_perc': 90
}

# ----------------------------
# Helper Function: simulate_spread
# ----------------------------
def simulate_spread(mean1, perc1, mean2, perc2, series1_id, series2_id,
                    full_corr, n_samples=1000, factor=55):
    """
    Simulate two forecast distributions and return the simulated spread.

    Parameters:
      mean1, mean2 : list-like, where the first element is the forecast mean.
      perc1, perc2 : list-like, with four values corresponding to the percentiles 
                     (p10, p25, p75, p90). These are used to approximate the sigma.
      series1_id, series2_id: Strings representing the full identifier (e.g. 'de_base').
      full_corr: A pandas DataFrame containing the full correlation matrix.
      n_samples: Number of simulation samples.
      factor: Multiplicative factor for the spread.
      
    Returns:
      spread: A numpy array of simulated spreads.
    """
    # Extract the correlation for the two series from the full correlation matrix.
    r = full_corr.loc[series1_id, series2_id]
    # Construct a bivariate correlation matrix.
    corr_matrix = np.array([[1.0, r],
                            [r, 1.0]])
    
    # Estimate sigma for each series using p10 and p90 (assuming a normal distribution).
    sigma1 = (perc1[-1] - perc1[0]) / (2 * 1.28)
    sigma2 = (perc2[-1] - perc2[0]) / (2 * 1.28)
    
    # Generate bivariate standard normal samples according to the constructed correlation.
    norm_samples = np.random.multivariate_normal(mean=[0, 0], cov=corr_matrix, size=n_samples)
    
    # Transform these samples to uniform variables.
    u = stats.norm.cdf(norm_samples)
    
    # Use the inverse CDF to transform the uniforms into the desired forecast distributions.
    sim1 = stats.norm.ppf(u[:, 0], loc=mean1[0], scale=sigma1)
    sim2 = stats.norm.ppf(u[:, 1], loc=mean2[0], scale=sigma2)
    
    # Calculate the spread and apply the scaling factor.
    spread = (sim1 - sim2) * factor
    return spread

# ----------------------------
# Simulation Structures
# ----------------------------
# We assume fcst_dict is a nested dictionary structured as:
# fcst_dict[market][period][del_type] = {'perc': [...], 'mean': [...]}
# For example, fcst_dict['de']['W_2']['base'] holds forecasts for the 'de' market in period 'W_2' and delivery type 'base'.
#
# The column identifiers for the full_corr lookup will be constructed as f"{market}_{del_type}"

# --- Simulation 1: Same Market, Same del_type Between Different Periods ---
sim_results_same_market_same_del = {}  # Structure: market -> del_type -> ((period1, period2): spread array)

for market, periods_data in fcst_dict.items():
    sim_results_same_market_same_del[market] = {}
    period_keys = list(periods_data.keys())
    # Assume each period has the same delivery types.
    if period_keys:
        del_types = list(periods_data[period_keys[0]].keys())
    else:
        continue

    for del_type in del_types:
        sim_results_same_market_same_del[market][del_type] = {}
        # Build the full series identifier, e.g. 'de_base' or 'de_peak'
        series_id = f"{market}_{del_type}"
        # Loop over all unique pairs of periods.
        for p1, p2 in itertools.combinations(period_keys, 2):
            fcst1 = fcst_dict[market][p1][del_type]
            fcst2 = fcst_dict[market][p2][del_type]
            if fcst1['perc'] is not None and fcst2['perc'] is not None:
                spread = simulate_spread(fcst1['mean'], fcst1['perc'], 
                                         fcst2['mean'], fcst2['perc'],
                                         series_id, series_id, full_corr,
                                         n_samples, spread_factor)
                sim_results_same_market_same_del[market][del_type][(p1, p2)] = spread

# --- Simulation 2: Same Market, Different del_types Within the Same Period ---
sim_results_same_market_diff_del = {}  # Structure: market -> period -> ((del_type1, del_type2): spread array)

for market, periods_data in fcst_dict.items():
    sim_results_same_market_diff_del[market] = {}
    for period, del_data in periods_data.items():
        sim_results_same_market_diff_del[market][period] = {}
        del_types = list(del_data.keys())
        for d1, d2 in itertools.combinations(del_types, 2):
            fcst1 = fcst_dict[market][period][d1]
            fcst2 = fcst_dict[market][period][d2]
            if fcst1['perc'] is not None and fcst2['perc'] is not None:
                # Build the full series identifiers.
                series_id1 = f"{market}_{d1}"
                series_id2 = f"{market}_{d2}"
                spread = simulate_spread(fcst1['mean'], fcst1['perc'],
                                         fcst2['mean'], fcst2['perc'],
                                         series_id1, series_id2, full_corr,
                                         n_samples, spread_factor)
                sim_results_same_market_diff_del[market][period][(d1, d2)] = spread

# --- Simulation 3: Across Markets 'de' and 'fr' for All (Period, del_type) Combinations ---
sim_results_de_fr = {}  # Structure: ((market1, period1, del_type1), (market2, period2, del_type2)) -> spread

markets_to_compare = ['de', 'fr']
# Proceed only if both markets are available in fcst_dict.
if all(m in fcst_dict for m in markets_to_compare):
    for period_de, data_de in fcst_dict['de'].items():
        for del_de, fcst_de in data_de.items():
            for period_fr, data_fr in fcst_dict['fr'].items():
                for del_fr, fcst_fr in data_fr.items():
                    key = (('de', period_de, del_de), ('fr', period_fr, del_fr))
                    if fcst_de['perc'] is not None and fcst_fr['perc'] is not None:
                        series_id_de = f"de_{del_de}"
                        series_id_fr = f"fr_{del_fr}"
                        spread = simulate_spread(fcst_de['mean'], fcst_de['perc'],
                                                 fcst_fr['mean'], fcst_fr['perc'],
                                                 series_id_de, series_id_fr, full_corr,
                                                 n_samples, spread_factor)
                        sim_results_de_fr[key] = spread

# ----------------------------
# (Optional) Function to Print Simulation Statistics
# ----------------------------
def print_simulation_stats(spread, description=""):
    print(description)
    print("Mean Simulation Spread:", np.mean(spread))
    print("Spread Percentiles (10th, 25th, 75th, 90th):", 
          np.percentile(spread, [10, 25, 75, 90]))
    # Calculate the percentile rank for the defined score.
    percentile = stats.percentileofscore(spread, score, kind='rank')
    print(f"Spread value {score} is at the {percentile:.2f}th percentile.\n")

# ----------------------------
# Output the Simulation Results
# ----------------------------
print("=== Simulation 1: Same Market, Same del_type Across Different Periods ===")
for market, del_data in sim_results_same_market_same_del.items():
    for del_type, period_pairs in del_data.items():
        for period_pair, spread in period_pairs.items():
            desc = f"Market: {market}, del_type: {del_type}, Periods: {period_pair}"
            print_simulation_stats(spread, description=desc)

print("=== Simulation 2: Same Market, Different del_types Within the Same Period ===")
for market, period_data in sim_results_same_market_diff_del.items():
    for period, pair_data in period_data.items():
        for del_pair, spread in pair_data.items():
            desc = f"Market: {market}, Period: {period}, del_types: {del_pair}"
            print_simulation_stats(spread, description=desc)

print("=== Simulation 3: Across Markets 'de' and 'fr' for All (Period, del_type) Combinations ===")
for key, spread in sim_results_de_fr.items():
    desc = f"Spread between {key[0]} and {key[1]}"
    print_simulation_stats(spread, description=desc)

=== Simulation 1: Same Market, Same del_type Across Different Periods ===
Market: de, del_type: base, Periods: ('Wknd_0', 'Wknd_1')
Mean Simulation Spread: 19.058567876933925
Spread Percentiles (10th, 25th, 75th, 90th): [-31.03603691  -7.57874094  45.70821522  71.73569838]
Spread value -5 is at the 27.80th percentile.

Market: de, del_type: base, Periods: ('Wknd_0', 'W_1')
Mean Simulation Spread: -9.965670975439066
Spread Percentiles (10th, 25th, 75th, 90th): [-61.52123221 -36.04296212  16.80450527  38.02430025]
Spread value -5 is at the 54.10th percentile.

Market: de, del_type: base, Periods: ('Wknd_0', 'W_2')
Mean Simulation Spread: -13.48367092788576
Spread Percentiles (10th, 25th, 75th, 90th): [-49.87114899 -32.34985819   5.21382465  22.15100999]
Spread value -5 is at the 61.50th percentile.

Market: de, del_type: base, Periods: ('Wknd_0', 'W_3')
Mean Simulation Spread: -21.631078756994203
Spread Percentiles (10th, 25th, 75th, 90th): [-56.99960157 -39.78740512  -2.53059769  14.731

In [50]:
# We assume that the following simulation results dictionaries exist from your simulations:
#   sim_results_same_market_same_del
#   sim_results_same_market_diff_del
#   sim_results_de_fr
#
# And that fcst_dict has the structure:
#   fcst_dict[market][period][del_type] = {'perc': [p10, p25, p75, p90], 'mean': [forecast_mean]}
#
# Also, these variables are already defined:
#   spread_factor (e.g., 55)
#
# The following code builds a list of dictionaries, one per simulation result, and then creates a DataFrame.

results = []

# --- Simulation 1: Same market, same del_type, different periods ---
# sim_results_same_market_same_del is structured as:
#   sim_results_same_market_same_del[market][del_type][(p1, p2)] = simulated spread (numpy array)
for market, del_dict in sim_results_same_market_same_del.items():
    for del_type, period_pair_dict in del_dict.items():
        for (p1, p2), spread in period_pair_dict.items():
            # Retrieve forecast information for each leg
            fcst1 = fcst_dict[market][p1][del_type]
            fcst2 = fcst_dict[market][p2][del_type]
            # Compute the forecast spread mean as defined (using the first value of forecast means)
            forecast_spread_mean = (fcst1['mean'][0] - fcst2['mean'][0]) * spread_factor
            simulated_spread_mean = np.mean(spread)
            sim_p10, sim_p25, sim_p75, sim_p90 = np.percentile(spread, [10, 25, 75, 90])
            
            results.append({
                "leg1_market": market,
                "leg1_period": p1,
                "leg1_deltype": del_type,
                "leg2_market": market,
                "leg2_period": p2,
                "leg2_deltype": del_type,
                "forecast_mean": forecast_spread_mean,
                "simulation_mean": simulated_spread_mean,
                "10th_perc": sim_p10,
                "25th_perc": sim_p25,
                "75th_perc": sim_p75,
                "90th_perc": sim_p90,
                "spread_type": "time_spread"   # Mark as time spread (different periods)
            })

# --- Simulation 2: Same market and period, different delivery types ---
# sim_results_same_market_diff_del is structured as:
#   sim_results_same_market_diff_del[market][period][(d1, d2)] = simulated spread (numpy array)
for market, period_dict in sim_results_same_market_diff_del.items():
    for period, del_pair_dict in period_dict.items():
        for (d1, d2), spread in del_pair_dict.items():
            fcst1 = fcst_dict[market][period][d1]
            fcst2 = fcst_dict[market][period][d2]
            forecast_spread_mean = (fcst1['mean'][0] - fcst2['mean'][0]) * spread_factor
            simulated_spread_mean = np.mean(spread)
            sim_p10, sim_p25, sim_p75, sim_p90 = np.percentile(spread, [10, 25, 75, 90])
            
            results.append({
                "leg1_market": market,
                "leg1_period": period,
                "leg1_deltype": d1,
                "leg2_market": market,
                "leg2_period": period,
                "leg2_deltype": d2,
                "forecast_mean": forecast_spread_mean,
                "simulation_mean": simulated_spread_mean,
                "10th_perc": sim_p10,
                "25th_perc": sim_p25,
                "75th_perc": sim_p75,
                "90th_perc": sim_p90,
                "spread_type": "base_peak_spread"   # Mark as base_peak spread (different del_types)
            })

# --- Simulation 3: Across markets (e.g. 'de' vs 'fr') ---
# sim_results_de_fr is structured as:
#   sim_results_de_fr[((market1, period1, del_type1), (market2, period2, del_type2))] = simulated spread (numpy array)
for key, spread in sim_results_de_fr.items():
    # Unpack the tuple key: ((market1, period1, del_type1), (market2, period2, del_type2))
    (m1, p1, d1), (m2, p2, d2) = key
    fcst1 = fcst_dict[m1][p1][d1]
    fcst2 = fcst_dict[m2][p2][d2]
    forecast_spread_mean = (fcst1['mean'][0] - fcst2['mean'][0]) * spread_factor
    simulated_spread_mean = np.mean(spread)
    sim_p10, sim_p25, sim_p75, sim_p90 = np.percentile(spread, [10, 25, 75, 90])
    
    results.append({
        "leg1_market": m1,
        "leg1_period": p1,
        "leg1_deltype": d1,
        "leg2_market": m2,
        "leg2_period": p2,
        "leg2_deltype": d2,
        "forecast_mean": forecast_spread_mean,
        "simulation_mean": simulated_spread_mean,
        "10th_perc": sim_p10,
        "25th_perc": sim_p25,
        "75th_perc": sim_p75,
        "90th_perc": sim_p90,
        "spread_type": "de_fr"   # Mark as de_fr (across markets)
    })

# Create a DataFrame from the aggregated results.
df_simulation_summary = pd.DataFrame(results)
# Create a list of the new column order: first "spread_type", then the rest of the columns.
cols = ['spread_type'] + [col for col in df_simulation_summary.columns if col != 'spread_type']
# Reindex the DataFrame with the new column order.
df_simulation_summary = df_simulation_summary[cols]

# (Optional) Save the DataFrame to disk
round(df_simulation_summary,2).to_excel(r"C:\Users\krajcovic\Documents\temp\simulation_summary.xlsx", index=False)

# Display the DataFrame
print(df_simulation_summary)

     spread_type leg1_market leg1_period leg1_deltype leg2_market leg2_period  \
0    time_spread          de      Wknd_0         base          de      Wknd_1   
1    time_spread          de      Wknd_0         base          de         W_1   
2    time_spread          de      Wknd_0         base          de         W_2   
3    time_spread          de      Wknd_0         base          de         W_3   
4    time_spread          de      Wknd_1         base          de         W_1   
..           ...         ...         ...          ...         ...         ...   
195        de_fr          de         W_3         peak          fr         W_1   
196        de_fr          de         W_3         peak          fr         W_2   
197        de_fr          de         W_3         peak          fr         W_2   
198        de_fr          de         W_3         peak          fr         W_3   
199        de_fr          de         W_3         peak          fr         W_3   

    leg2_deltype  forecast_